# 02 · Feature engineering, leakage ablation, feature selection & split strategy

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
from src import config
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
M, F = config.METRICS_DIR, config.FIG_DIR
show = lambda name, w=900: display(Image(filename=str(F / name), width=w))

In [2]:
from src.features import FEATURE_DOCS, add_engineered_features
from src.data_loader import load_clean
from src.preprocessing import drop_exact_duplicates, add_per_ha_target
df = add_engineered_features(add_per_ha_target(drop_exact_duplicates(load_clean())))
pd.DataFrame({'definition': FEATURE_DOCS, 'n_unique': df[list(FEATURE_DOCS)].nunique(),
              'corr_with_yield_per_ha': df[list(FEATURE_DOCS)].corrwith(df[config.TARGET_PER_HA]).round(3)})

,definition,n_unique,corr_with_yield_per_ha
n_applied_kg,Nitrogen applied = 0.46*Urea + 0.18*DAP (kg),6,0.595
n_per_ha,"Nitrogen intensity, kg N / ha",1,0.000
p2o5_per_ha,Phosphate intensity = 0.46*DAP / ha (kg P2O5/ha),1,-0.000
k2o_per_ha,Potash intensity = 0.60*Potash / ha (kg K2O/ha),1,-0.000
n_to_k_ratio,N : K2O balance (nutrient ratio),1,-0.000
total_rain_mm,"Season rainfall, sum of the 4 crop-stage windo...",3,-0.008
total_irrigation_mm,Season 'AI' water (source abbreviation; comple...,3,0.008
rain_share,Fraction of season water supplied by rain = ra...,3,-0.008
tmean_season,Mean of (Tmin+Tmax)/2 over the 4 thirty-day wi...,6,0.006
diurnal_range_mean,Mean (Tmax - Tmin) over the season (deg C),5,-0.012


## Feature selection (fitted on the training split only)

In [3]:
sel = json.loads((M/'feature_selection.json').read_text())
print({k: sel[k] for k in ['n_source_predictors','n_leakage_removed','n_engineered','n_encoded_features_before_selection','n_selected','reduction_pct']})
pd.Series(sel['dropped'], name='reason').to_frame()

{'n_source_predictors': 44, 'n_leakage_removed': 1, 'n_engineered': 15, 'n_encoded_features_before_selection': 69, 'n_selected': 44, 'reduction_pct': 36.2}


,reason
n_per_ha,constant
p2o5_per_ha,constant
k2o_per_ha,constant
n_to_k_ratio,constant
seed_rate_kg,|r|>=0.99 with hectares (r=1.0000)
lp_mainfield_t,|r|>=0.99 with hectares (r=1.0000)
nursery_area_cents,|r|>=0.99 with hectares (r=1.0000)
lp_nursery_t,|r|>=0.99 with hectares (r=1.0000)
dap_20d,|r|>=0.99 with hectares (r=1.0000)
thiobencarb_28d,|r|>=0.99 with hectares (r=1.0000)


In [4]:
pd.read_csv(M/'feature_selection_cv_check.csv')

,target,feature_set,n_features,cv_R2,cv_RMSE
0,total,all encoded features,69,0.9911,873.8117
1,total,selected features,44,0.9911,873.8117
2,per_ha,all encoded features,69,0.4501,212.8858
3,per_ha,selected features,44,0.4501,212.8858


### Mutual information vs LightGBM gain (per-hectare target)

In [5]:
pd.read_csv(M/'feature_scores_per_ha.csv', index_col=0).head(20)

,mutual_information,lgbm_gain_share
hectares,0.41925,0.84666
rain_d30_50_mm,0.02221,0.00239
soil_type=clay,0.02022,0.02264
variety=ponmani,0.01951,0.01584
agriblock=Cuddalore,0.01460,0.00000
variety=delux ponni,0.01162,0.02667
rain_d51_70_mm,0.00931,0.00000
wind_kn_d31_60,0.00916,0.00000
agriblock=Kallakurichi,0.00850,0.00000
variety=CO_43,0.00802,0.00709


In [6]:
pd.read_csv(M/'feature_scores_total.csv', index_col=0).head(10)

,mutual_information,lgbm_gain_share
hectares,1.69040,0.99783
rain_d30_50_mm,0.03763,0.00002
variety=delux ponni,0.03697,0.00050
tmax_d31_60,0.01580,0.00006
agriblock=Panruti,0.01559,0.00000
rain_d51_70_mm,0.01113,0.00000
variety=CO_43,0.01103,0.00007
tmax_d91_120,0.01091,0.00000
wind_dir_d91_120_sin,0.00977,0.00000
rh_d61_90,0.00872,0.00000


## Leakage ablation (5-fold CV, default LightGBM)

In [7]:
pd.read_csv(M/'leakage_ablation.csv')

,target,setting,n_rows,cv_R2,cv_RMSE,cv_MAE
0,total,raw incl. duplicates + trash (naive),2789,0.9917,835.0512,596.9199
1,total,deduplicated + trash,2338,0.9905,900.5056,648.7318
2,total,"deduplicated, no trash (all features)",2338,0.9905,898.5019,647.6258
3,total,compact: hectares+agriblock+variety+soil+nursery,2338,0.9907,889.5688,642.9338
4,total,hectares only (OLS),2338,0.9887,983.3141,742.6209
5,per_ha,raw incl. duplicates + trash (naive),2789,0.4875,202.5537,157.9712
6,per_ha,deduplicated + trash,2338,0.4301,217.9454,172.2071
7,per_ha,"deduplicated, no trash (all features)",2338,0.4348,217.0853,171.4342
8,per_ha,compact: hectares+agriblock+variety+soil+nursery,2338,0.4475,214.6516,170.0090
9,per_ha,hectares only (OLS),2338,0.3522,232.5676,184.0705


## Split-strategy sensitivity (training data)

In [8]:
pd.read_csv(M/'split_sensitivity.csv')

,target,scheme,cv_R2,cv_R2_std,cv_RMSE
0,total,Random 5-fold,0.9911,0.0012,873.8117
1,total,Leave-one-agriblock-out (6 folds),0.9911,0.0017,867.8851
2,total,Group by management config (5 folds),0.9909,0.0004,882.8694
3,per_ha,Random 5-fold,0.4501,0.0593,212.8858
4,per_ha,Leave-one-agriblock-out (6 folds),0.4832,0.0620,206.3207
5,per_ha,Group by management config (5 folds),0.4534,0.0386,212.1113


## Findings

* 15 agronomic features were engineered and documented. **4 are constant** (N, P₂O₅, K₂O per ha and the N:K ratio — every farm uses the same dose), **5 are ≥0.99-correlated duplicates** of existing columns (N applied ≡ hectares; season rain/irrigation/rain-share ≡ one rain column; GDD ≡ mean temperature). The 6 survivors carry ~0 SHAP importance. Feature engineering therefore **did not add predictive signal** on this dataset — reported, not hidden.
* **Feature selection** (constant + |r|≥0.99 redundancy filter, fit on train only): 69 encoded features → **44 (36.2% reduction)** with **identical CV score** (LightGBM CV R² 0.4501 vs 0.4501 per-ha; 0.9911 vs 0.9911 total). MI and LightGBM gain both rank `hectares` far above everything else.
* **Leakage ablation** (CV, LightGBM): keeping the duplicate rows inflates per-ha R² from 0.435 → 0.488 — duplicate leakage is real and measurable. Trash adds nothing here. A compact model with only *hectares + agriblock + variety + soil + nursery* (5 source variables) matches the full feature set (0.448 vs 0.435 per-ha; 0.9907 vs 0.9905 total) — the 44 columns hold about 5 variables of information.
* **Split decision**: no time column → time split impossible. Random 5-fold, leave-one-agriblock-out and group-by-configuration CV give consistent per-ha R² (0.45 / 0.48 / 0.45), so random splitting does not leak through location here. Final protocol: deduplicate → stratified (hectares × agriblock) 80/20 hold-out → 5-fold CV on train for all model selection & tuning.
